In [0]:

# Databricks notebook source
# MAGIC %md
# MAGIC # Camada Ingestao: Extracao de APIs (Web to Bronze)
# MAGIC **Projeto:** Aero Clima | **Camada:** Ingestion | **Versao:** 2.2
# MAGIC
# MAGIC Responsavel por autenticar nas APIs, coletar dados em tempo real e salvar 
# MAGIC os JSONs brutos diretamente no Volume do Unity Catalog.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Configuracao e Credenciais

# COMMAND ----------

import requests
import json
import time
from datetime import datetime

# Caminho de destino no Lakehouse
BRONZE_PATH = "/Volumes/workspace/default/camada_bronze/"

OPENSKY_CLIENT_ID = dbutils.secrets.get(
    scope="aero_clima_secrets",
    key="opensky_client_id"
)

OPENSKY_CLIENT_SECRET = dbutils.secrets.get(
    scope="aero_clima_secrets",
    key="opensky_client_secret"
)

OPENWEATHER_API_KEY = dbutils.secrets.get(
    scope="aero_clima_secrets",
    key="openweather_api_key"
)

# Parametros de Geolocalizacao (Corredor Sul-Sudeste)
BBOX_VOOS = "lamin=-30.5&lomin=-52.0&lamax=-23.0&lomax=-45.5"

# Aeroportos de Monitoramento
AEROPORTOS = {
    "GRU": (-23.43, -46.47), 
    "CGH": (-23.62, -46.65), 
    "POA": (-29.99, -51.17)
}

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Gestao de Autenticacao (TokenManager)

# COMMAND ----------

class TokenManager:
    """Gerencia a geracao e renovacao automatica do token OAuth2 para a OpenSky."""
    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.token = None
        self.url = "https://opensky-network.org/auth/realms/opensky/protocol/openid-connect/token"

    def get_token(self):
        payload = {
            'grant_type': 'client_credentials',
            'client_id': self.client_id,
            'client_secret': self.client_secret
        }
        try:
            response = requests.post(self.url, data=payload, timeout=15)
            if response.status_code == 200:
                self.token = response.json().get('access_token')
                print("Log: Token OAuth2 gerado com sucesso.")
                return self.token
            else:
                print(f"Falha na autenticacao: {response.status_code} - {response.text}")
                return None
        except Exception as e:
            print(f"Erro de conexao no TokenManager: {e}")
            return None

# Instancia o gerenciador de tokens
auth_manager = TokenManager(OPENSKY_CLIENT_ID, OPENSKY_CLIENT_SECRET)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Funcoes de Persistencia

# COMMAND ----------

def salvar_no_volume(dados, prefixo):
    """Persiste o dicionario em formato JSON no Volume Delta."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"{BRONZE_PATH}{prefixo}_{timestamp}.json"
    
    # dbutils.fs.put salva o dado de forma distribuida no storage da nuvem
    dbutils.fs.put(file_path, json.dumps(dados), overwrite=True)
    print(f"Log: Persistencia concluida -> {file_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Execucao: Ingestao de Voos (OpenSky)

# COMMAND ----------

token = auth_manager.get_token()

if token:
    headers = {'Authorization': f'Bearer {token}'}
    url_voos = f"https://opensky-network.org/api/states/all?{BBOX_VOOS}"
    
    print("Coletando estados de voos em tempo real...")
    try:
        res_voos = requests.get(url_voos, headers=headers, timeout=20)
        if res_voos.status_code == 200:
            salvar_no_volume(res_voos.json(), "flights")
        else:
            print(f"Erro na API OpenSky: {res_voos.status_code}")
    except Exception as e:
        print(f"Falha na requisicao de voos: {e}")
else:
    print("Cancelando ingestao de voos: Falha na geracao do Bearer Token.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Execucao: Ingestao de Clima (OpenWeather)

# COMMAND ----------

print("Coletando dados meteorologicos para os aeroportos monitorados...")
weather_payload = {}

for aero, coords in AEROPORTOS.items():
    url_w = f"https://api.openweathermap.org/data/2.5/weather?lat={coords[0]}&lon={coords[1]}&appid={OPENWEATHER_API_KEY}&units=metric"
    try:
        res_w = requests.get(url_w, timeout=10)
        if res_w.status_code == 200:
            weather_payload[aero] = res_w.json()
        else:
            print(f"Alerta: Falha na coleta de clima para {aero}")
    except Exception as e:
        print(f"Erro de conexao OpenWeather ({aero}): {e}")

if weather_payload:
    salvar_no_volume(weather_payload, "weather")

print("Pipeline de Ingestao finalizado com sucesso.")

Falha na autenticacao: 403 - <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>
<hr><center>nginx</center>
</body>
</html>

Cancelando ingestao de voos: Falha na geracao do Bearer Token.
Coletando dados meteorologicos para os aeroportos monitorados...


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Camada Bronze: Ingestao e Validacao de Dados Brutos
# MAGIC **Projeto:** Aero Clima | **Camada:** Bronze | **Versao:** 2.1
# MAGIC
# MAGIC Responsavel pela leitura dos arquivos JSON brutos depositados pelo pipeline local
# MAGIC e pela validacao estrutural basica antes da promocao dos dados a Camada Prata.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Configuracao e Constantes

# COMMAND ----------

# Caminhos centralizados de infraestrutura
BRONZE_PATH   = "/Volumes/workspace/default/camada_bronze"
CATALOG       = "workspace"
DATABASE      = "default"

FLIGHTS_GLOB  = f"{BRONZE_PATH}/flights_*.json"
WEATHER_GLOB  = f"{BRONZE_PATH}/weather_*.json"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Diagnostico do Volume Bronze

# COMMAND ----------

files = dbutils.fs.ls(BRONZE_PATH)

if not files:
    raise FileNotFoundError(
        f"CRITICO: Nenhum arquivo encontrado em '{BRONZE_PATH}'. "
        "Verifique a execucao do pipeline de ingestao (main.py)."
    )

print(f"Diagnostico: {len(files)} arquivo(s) mapeado(s) no diretorio Bronze.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Leitura dos Dados Brutos de Voos

# COMMAND ----------

df_voos_raw = (
    spark.read
    .option("multiline", "true")
    .json(FLIGHTS_GLOB)
)

# Validacao estrutural de metadados
expected_cols = {"time", "states"}
actual_cols   = set(df_voos_raw.columns)
missing       = expected_cols - actual_cols

if missing:
    raise ValueError(
        f"FALHA DE CONTRATO: Schema inesperado nos dados de voos. Colunas ausentes: {missing}."
    )

print("Leitura de dados de voos (OpenSky) concluida com sucesso. Schema validado.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Leitura dos Dados Brutos de Clima

# COMMAND ----------

df_clima_raw = (
    spark.read
    .option("multiline", "true")
    .json(WEATHER_GLOB)
)

# Validacao de presenca das chaves primarias esperadas (Aeroportos monitorados)
expected_airports = {"CGH", "GRU", "POA"}
actual_airports   = set(df_clima_raw.columns)
missing_airports  = expected_airports - actual_airports

if missing_airports:
    raise ValueError(
        f"FALHA DE CONTRATO: Schema inesperado nos dados de clima. Aeroportos ausentes: {missing_airports}."
    )

print("Leitura de dados de clima (OpenWeather) concluida com sucesso. Schema validado.")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Camada Prata: Transformacao e Limpeza de Dados
# MAGIC **Projeto:** Aero Clima | **Camada:** Silver | **Versao:** 3.0 (Prod Ready)
# MAGIC
# MAGIC Responsavel por:
# MAGIC - Normalizar dados semi-estruturados (explode / unpivot)
# MAGIC - Aplicar tipagem forte (schema enforcement)
# MAGIC - Garantir consistencia entre cargas
# MAGIC - Preparar dados para consumo analitico

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Imports e Constantes

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG  = "workspace"
DATABASE = "default"

TABLE_VOOS_SILVER   = f"{CATALOG}.{DATABASE}.voos_silver"
TABLE_CLIMA_SILVER  = f"{CATALOG}.{DATABASE}.clima_silver"

BRONZE_PATH  = "/Volumes/workspace/default/camada_bronze"
FLIGHTS_GLOB = f"{BRONZE_PATH}/flights_*.json"
WEATHER_GLOB = f"{BRONZE_PATH}/weather_*.json"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Schema Definido (Clima - Padronizacao)

# COMMAND ----------

weather_schema = StructType([
    StructField("dt", LongType()),
    StructField("visibility", LongType()),
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("humidity", LongType())
    ])),
    StructField("wind", StructType([
        StructField("speed", DoubleType()),
        StructField("deg", LongType())
    ])),
    StructField("weather", ArrayType(
        StructType([
            StructField("main", StringType()),
            StructField("description", StringType())
        ])
    ))
])

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Processamento: Voos (Bronze → Silver)

# COMMAND ----------

df_voos_raw = spark.read.option("multiline", "true").json(FLIGHTS_GLOB)

df_voos_silver = (
    df_voos_raw
    .select(
        F.col("time").cast(LongType()).alias("timestamp_unix"),
        F.explode("states").alias("v")
    )
    .select(
        F.to_timestamp("timestamp_unix").alias("timestamp_coleta"),
        F.col("v")[0].alias("icao24"),
        F.trim(F.col("v")[1]).alias("callsign"),
        F.col("v")[2].alias("origin_country"),
        F.col("v")[5].cast(FloatType()).alias("longitude"),
        F.col("v")[6].cast(FloatType()).alias("latitude"),
        F.col("v")[8].cast(BooleanType()).alias("on_ground"),
        F.col("v")[9].cast(FloatType()).alias("velocity_ms"),
        F.col("v")[10].cast(FloatType()).alias("true_track_deg"),
        F.col("v")[11].cast(FloatType()).alias("vertical_rate_ms"),
    )
    .filter(F.col("icao24").isNotNull())
    .dropDuplicates(["icao24", "timestamp_coleta"])
)

# COMMAND ----------

(
    df_voos_silver.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "false")
    .saveAsTable(TABLE_VOOS_SILVER)
)

print(f"[OK] {TABLE_VOOS_SILVER} atualizada.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Processamento: Clima (Bronze → Silver)

# COMMAND ----------

df_clima_raw = spark.read.option("multiline", "true").json(WEATHER_GLOB)

# Normalizacao segura usando JSON (resolve schema inconsistente)
df_clima_normalizado = (
    df_clima_raw
    .select(
        F.explode(
            F.array(
                F.struct(F.lit("CGH").alias("aeroporto"), F.to_json("CGH").alias("payload")),
                F.struct(F.lit("GRU").alias("aeroporto"), F.to_json("GRU").alias("payload")),
                F.struct(F.lit("POA").alias("aeroporto"), F.to_json("POA").alias("payload")),
            )
        ).alias("dado")
    )
    .select(
        F.col("dado.aeroporto").alias("aeroporto_sigla"),
        F.from_json("dado.payload", weather_schema).alias("payload")
    )
)

df_clima_silver = (
    df_clima_normalizado
    .select(
        "aeroporto_sigla",
        F.to_timestamp("payload.dt").alias("timestamp_coleta"),
        F.col("payload.main.temp").cast(FloatType()).alias("temperatura_c"),
        F.col("payload.main.humidity").cast(FloatType()).alias("umidade_pct"),
        F.col("payload.wind.speed").cast(FloatType()).alias("vento_velocidade_ms"),
        F.col("payload.wind.deg").cast(FloatType()).alias("vento_direcao_graus"),
        F.col("payload.weather")[0]["main"].alias("condicao_climatica"),
        F.col("payload.weather")[0]["description"].alias("descricao_climatica"),
        F.col("payload.visibility").cast(FloatType()).alias("visibilidade_m"),
    )
    .filter(
        F.col("aeroporto_sigla").isNotNull() &
        F.col("timestamp_coleta").isNotNull()
    )
    .dropDuplicates(["aeroporto_sigla", "timestamp_coleta"])
)

# COMMAND ----------

(
    df_clima_silver.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "false")
    .saveAsTable(TABLE_CLIMA_SILVER)
)

print(f"[OK] {TABLE_CLIMA_SILVER} atualizada.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Validacoes Basicas (Data Quality)

# COMMAND ----------

print("Voos:", df_voos_silver.count())
print("Clima:", df_clima_silver.count())

display(df_clima_silver.limit(10))
display(df_voos_silver.limit(10))

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Camada Ouro: Enriquecimento, SLA e Monitoramento Operacional
# MAGIC **Projeto:** Aero Clima | **Camada:** Gold | **Versao:** 2.1
# MAGIC
# MAGIC Implementacao de business logic, cruzamento espacial de dimensoes e avaliacao de
# MAGIC nivel de servico (SLA) com base em condicoes meteorologicas corporativas.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Imports e Parametrizacao de Negocio

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

CATALOG  = "workspace"
DATABASE = "default"

TABLE_VOOS_SILVER  = f"{CATALOG}.{DATABASE}.voos_silver"
TABLE_CLIMA_SILVER = f"{CATALOG}.{DATABASE}.clima_silver"
TABLE_GOLD_MONITOR = f"{CATALOG}.{DATABASE}.monitor_operacional_gold"

# Matriz de tolerância operacional (Limites de Vento em m/s)
WIND_LIMITS = {
    "CGH": {"alto": 9.0,  "medio": 6.0},
    "GRU": {"alto": 13.0, "medio": 9.0},
    "POA": {"alto": 11.0, "medio": 7.5},
}

HIGH_RISK_CONDITIONS = ["Thunderstorm", "Snow", "Tornado", "Squall"]
MED_RISK_CONDITIONS  = ["Rain", "Drizzle", "Fog", "Mist", "Haze"]

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Geofencing e Integracao Relacional

# COMMAND ----------

df_voos  = spark.read.table(TABLE_VOOS_SILVER)
df_clima = spark.read.table(TABLE_CLIMA_SILVER)

# Atribuicao de quadrante baseada em latitude para o corredor Sul-Sudeste
df_voos_geo = df_voos.withColumn(
    "aeroporto_proximo",
    F.when(F.col("latitude") < -28.0, "POA")
     .when((F.col("latitude") >= -28.0) & (F.col("latitude") < -23.55), "CGH")
     .otherwise("GRU")
)

# Left join garante preservacao de registros aeronauticos independentemente da falha do sensor climatico
df_joined = df_voos_geo.join(
    df_clima.select(
        F.col("aeroporto_sigla"),
        F.col("timestamp_coleta").alias("ts_clima"),
        F.col("temperatura_c"),
        F.col("umidade_pct"),
        F.col("vento_velocidade_ms"),
        F.col("vento_direcao_graus"),
        F.col("condicao_climatica"),
        F.col("descricao_climatica"),
        F.col("visibilidade_m"),
    ),
    on=df_voos_geo["aeroporto_proximo"] == F.col("aeroporto_sigla"),
    how="left",
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Motor de Avaliacao de Risco

# COMMAND ----------

def build_risk_expression():
    """
    Funcao utilitaria que constroi o motor de regras de forma dinamica.
    Prioriza condicoes severas (High Risk) independentemente do fluxo regular.
    """
    expr = F.lit("NORMAL")

    for iata, limits in WIND_LIMITS.items():
        expr = (
            F.when(
                (F.col("aeroporto_proximo") == iata) & (F.col("vento_velocidade_ms") > limits["medio"]),
                "MEDIO"
            ).otherwise(expr)
        )

    expr = F.when(F.col("condicao_climatica").isin(MED_RISK_CONDITIONS), "MEDIO").otherwise(expr)

    for iata, limits in WIND_LIMITS.items():
        expr = (
            F.when(
                (F.col("aeroporto_proximo") == iata) & (F.col("vento_velocidade_ms") > limits["alto"]),
                "ALTO"
            ).otherwise(expr)
        )

    expr = F.when(F.col("condicao_climatica").isin(HIGH_RISK_CONDITIONS), "ALTO").otherwise(expr)
    return expr


df_gold = (
    df_joined
    .withColumn("nivel_risco", build_risk_expression())
    .withColumn(
        "status_sla",
        F.when(F.col("nivel_risco") == "ALTO",   "CRITICO")
         .when(F.col("nivel_risco") == "MEDIO",  "ATENCAO")
         .otherwise("OPERACIONAL")
    )
    .withColumn(
        "alerta_sla",
        F.when(F.col("nivel_risco") == "ALTO",  "INTERRUPCAO IMINENTE")
         .when(F.col("nivel_risco") == "MEDIO", "MONITORAR")
         .otherwise("OPERACAO NORMAL")
    )
    .withColumn("timestamp_processamento", F.current_timestamp())
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Persistencia da Visao Consolidada

# COMMAND ----------

df_monitor = df_gold.select(
    F.col("icao24").alias("id_aeronave"),
    F.trim(F.col("callsign")).alias("identificacao_voo"),
    F.col("origin_country").alias("pais_origem"),
    F.col("aeroporto_proximo").alias("aeroporto_monitorado"),
    F.col("on_ground").alias("em_solo"),
    F.round(F.col("velocity_ms"), 2).alias("velocidade_ms"),
    F.round(F.col("true_track_deg"), 1).alias("direcao_graus"),
    F.round(F.col("vertical_rate_ms"), 2).alias("taxa_vertical_ms"),
    F.col("condicao_climatica").alias("clima"),
    F.col("descricao_climatica").alias("clima_descricao"),
    F.round(F.col("temperatura_c"), 1).alias("temperatura_c"),
    F.round(F.col("umidade_pct"), 1).alias("umidade_pct"),
    F.round(F.col("vento_velocidade_ms"), 2).alias("vento_ms"),
    F.round(F.col("vento_direcao_graus"), 1).alias("vento_direcao_graus"),
    F.col("visibilidade_m").alias("visibilidade_m"),
    F.col("nivel_risco"),
    F.col("status_sla"),
    F.col("alerta_sla"),
    F.col("timestamp_coleta").alias("timestamp_voo"),
    F.col("ts_clima").alias("timestamp_clima"),
    F.col("timestamp_processamento"),
)

(
    df_monitor.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_GOLD_MONITOR)
)
print(f"Log: Tabela {TABLE_GOLD_MONITOR} consolidada e sobrescrita com exito.")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Modelagem Dimensional: Star Schema
# MAGIC **Projeto:** Aero Clima | **Camada:** Serving | **Versao:** 2.1
# MAGIC
# MAGIC Otimizacao do modelo relacional para arquitetura Kimball (Esquema Estrela),
# MAGIC preparando consumo analitico pelo Power BI via SQL Warehouse.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Imports e Constantes

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, DoubleType
)

CATALOG  = "workspace"
DATABASE = "default"

TABLE_GOLD_MONITOR = f"{CATALOG}.{DATABASE}.monitor_operacional_gold"
TABLE_DIM_AIRPORTS = f"{CATALOG}.{DATABASE}.dim_aeroportos"
TABLE_FACT_OPS     = f"{CATALOG}.{DATABASE}.fato_operacoes"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Manutencao de Dimensoes Estaticas

# COMMAND ----------

AIRPORTS_SCHEMA = StructType([
    StructField("sigla",            StringType(), nullable=False),
    StructField("nome_aeroporto",   StringType(), nullable=False),
    StructField("cidade",           StringType(), nullable=False),
    StructField("estado",           StringType(), nullable=False),
    StructField("latitude_ref",     DoubleType(), nullable=False),
    StructField("longitude_ref",    DoubleType(), nullable=False),
    StructField("limite_vento_alto_ms",  FloatType(), nullable=False),
    StructField("limite_vento_medio_ms", FloatType(), nullable=False),
])

airports_data = [
    ("CGH", "Congonhas",    "Sao Paulo",   "SP", -23.6261, -46.6564,  9.0,  6.0),
    ("GRU", "Guarulhos",    "Guarulhos",   "SP", -23.4356, -46.4731, 13.0,  9.0),
    ("POA", "Salgado Filho", "Porto Alegre","RS", -29.9944, -51.1713, 11.0,  7.5),
]

df_dim_aeroportos = spark.createDataFrame(airports_data, schema=AIRPORTS_SCHEMA)

(
    df_dim_aeroportos.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE_DIM_AIRPORTS)
)
print(f"Log: Dimensao {TABLE_DIM_AIRPORTS} atualizada.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Construcao da Tabela Fato de Operacoes

# COMMAND ----------

df_gold = spark.read.table(TABLE_GOLD_MONITOR)

df_fato = (
    df_gold
    # Utilizacao de hash SHA256 garante criacao de chave primaria deterministica,
    # prevenindo duplicidade de IDs em operacoes de insercao continua (append).
    .withColumn(
        "id_operacao",
        F.sha2(
            F.concat_ws("|",
                F.col("id_aeronave"),
                F.col("timestamp_processamento").cast("string")
            ),
            256
        )
    )
    .select(
        F.col("id_operacao"),                                         
        F.col("identificacao_voo").alias("fk_voo"),                    
        F.col("aeroporto_monitorado").alias("fk_aeroporto"),           
        F.col("timestamp_processamento").alias("timestamp_evento"),
        F.col("velocidade_ms").cast(FloatType()),
        F.col("taxa_vertical_ms").cast(FloatType()),
        F.col("direcao_graus").cast(FloatType()),
        F.col("em_solo"),
        F.col("temperatura_c").cast(FloatType()),
        F.col("umidade_pct").cast(FloatType()),
        F.col("vento_ms").cast(FloatType()),
        F.col("vento_direcao_graus").cast(FloatType()),
        F.col("visibilidade_m").cast(FloatType()),
        F.col("nivel_risco"),
        F.col("status_sla"),
        F.col("alerta_sla"),
    )
    .dropDuplicates(["id_operacao"])
)

# Alteracao Arquitetural: 
# Uso de "append" em vez de "overwrite" garante a retencao do log historico.
(
    df_fato.write
    .format("delta")
    .mode("append") 
    .option("mergeSchema", "true")
    .saveAsTable(TABLE_FACT_OPS)
)

print(f"Log: Dados operacionais anexados a Tabela Fato ({TABLE_FACT_OPS}) com exito.")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Utilitario: Carga Historica e Dados Sinteticos
# MAGIC **Projeto:** Aero Clima | **Utilitario** | **Versao:** 2.2
# MAGIC
# MAGIC Script de execucao unica (One-off) destinado a populacao da camada de visualizacao
# MAGIC com massa de dados representativa baseada em metricas reais.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Configuracao de Parametros

# COMMAND ----------

dbutils.widgets.text("dias_historico",   "7",    "Dias de historico a simular")
dbutils.widgets.text("fator_expansao",   "50",   "Multiplicador de linhas")
dbutils.widgets.text("seed_aleatorio",   "42",   "Seed para reprodutibilidade")

DIAS_HISTORICO = int(dbutils.widgets.get("dias_historico"))
FATOR_EXPANSAO = int(dbutils.widgets.get("fator_expansao"))
SEED           = int(dbutils.widgets.get("seed_aleatorio"))

CATALOG  = "workspace"
DATABASE = "default"
TABLE_FACT_OPS = f"{CATALOG}.{DATABASE}.fato_operacoes"

print(f"Iniciando procedimento de carga parametrizado para {DIAS_HISTORICO} dias.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Processamento

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

try:
    df_base = spark.read.table(TABLE_FACT_OPS)
except Exception as e:
    raise RuntimeError(f"Falha ao localizar a tabela Fato. Certifique-se de executar o pipeline previamente. Erro: {e}")

SEGUNDOS_TOTAL = DIAS_HISTORICO * 24 * 3600

df_historico = (
    df_base
    .repartition(min(FATOR_EXPANSAO, 200))
    .withColumn("replica_id", F.monotonically_increasing_id())
    .withColumn("segundos_atras", (F.rand(seed=SEED) * SEGUNDOS_TOTAL).cast("int"))
    .withColumn("timestamp_evento", F.expr(f"current_timestamp() - make_interval(0, 0, 0, 0, 0, 0, segundos_atras)"))
    
    # CORRECAO CRITICA: O cast(FloatType) deve envelopar toda a operacao matematica e o round
    .withColumn("vento_ms", F.round(F.col("vento_ms") * (F.rand(seed=SEED + 1) + 0.5), 2).cast(FloatType()))
    .withColumn("temperatura_c", F.round(F.col("temperatura_c") + (F.rand(seed=SEED + 2) * 5 - 2.5), 1).cast(FloatType()))
    .withColumn("umidade_pct", F.round(F.least(F.lit(100.0), F.greatest(F.lit(0.0), F.col("umidade_pct") + (F.rand(seed=SEED + 3) * 20 - 10))), 1).cast(FloatType()))
    
    .withColumn(
        "nivel_risco",
        F.when((F.col("fk_aeroporto") == "CGH") & (F.col("vento_ms") > 9.0),  "ALTO")
        .when((F.col("fk_aeroporto") == "GRU") & (F.col("vento_ms") > 13.0), "ALTO")
        .when((F.col("fk_aeroporto") == "POA") & (F.col("vento_ms") > 11.0), "ALTO")
        .when(F.col("vento_ms") > 6.0, "MEDIO")
        .otherwise("NORMAL")
    )
    .withColumn(
        "status_sla",
        F.when(F.col("nivel_risco") == "ALTO",  "CRITICO")
         .when(F.col("nivel_risco") == "MEDIO", "ATENCAO")
         .otherwise("OPERACIONAL")
    )
    .withColumn(
        "alerta_sla",
        F.when(F.col("nivel_risco") == "ALTO",  "INTERRUPCAO IMINENTE")
         .when(F.col("nivel_risco") == "MEDIO", "MONITORAR")
         .otherwise("OPERACAO NORMAL")
    )
    .withColumn(
        "id_operacao",
        F.sha2(F.concat_ws("|", F.col("fk_voo"), F.col("timestamp_evento").cast("string"), F.col("replica_id").cast("string")), 256)
    )
    .drop("segundos_atras", "replica_id")
    .dropDuplicates(["id_operacao"])
)

# COMMAND ----------

# Persistencia
(
    df_historico.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true") # Adicionado para garantir transicao suave de metadados
    .saveAsTable(TABLE_FACT_OPS)
)

print("Procedimento de injecao de carga historica finalizado com exito.")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data